# **CRAWLING BERITA DARI DETIK.COM**

In [1]:
import requests
from bs4 import BeautifulSoup
import time
import re
import string
import sys
import pandas as pd
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import random

# --- FUNGSI-FUNGSI BANTUAN SCRAPING ---
def print_progress(kategori, current_page, total_pages):
    """Menampilkan progress bar di konsol."""
    percent = (current_page / total_pages) * 100 if total_pages > 0 else 0
    bar_length = 20
    filled_length = int(bar_length * current_page // total_pages) if total_pages > 0 else 0
    bar = '█' * filled_length + '-' * (bar_length - filled_length)
    sys.stdout.write(f'\r{kategori} - Page {current_page}/{total_pages} [{bar}] {percent:.2f}%')
    sys.stdout.flush()
    if current_page == total_pages:
        sys.stdout.write('\n\n')

def get_session():
    """Membuat sesi permintaan dengan mekanisme percobaan ulang."""
    session = requests.Session()
    retry_strategy = Retry(
        total=5,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
        backoff_factor=1
    )
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount("https://", adapter)
    return session

def get_article_content_and_title(session, url):
    """Mengambil isi artikel dan judul dari URL."""
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/108.0.0.0 Safari/537.36'
    }
    try:
        r = session.get(url, headers=headers, timeout=15)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")

        title = get_article_title(soup)
        content_selectors = [
            "div.detail-konten", "div.news-detail__content", "div.itp_bodycontent",
            "div.content-text", "div.article-content", "div.text_area"
        ]

        paragraphs = []
        for selector in content_selectors:
            content_divs = soup.select(selector)
            if content_divs:
                for div in content_divs:
                    for p in div.find_all("p"):
                        text = p.get_text(strip=True)
                        if text and not text.lower().startswith("baca juga"):
                            paragraphs.append(text)
                if paragraphs:
                    break
        
        if not paragraphs:
            body_text = soup.find("article")
            if body_text:
                for p in body_text.find_all("p"):
                    text = p.get_text(strip=True)
                    if text and not text.lower().startswith("baca juga"):
                        paragraphs.append(text)

        content = " ".join(paragraphs)
        return title, content
    except requests.exceptions.RequestException as e:
        print(f"Error fetching {url}: {e}", file=sys.stderr)
        return "Judul Tidak Ditemukan", ""

def get_article_title(soup):
    """Mengambil judul artikel dari berbagai kemungkinan lokasi."""
    title_tag = soup.find("h1", class_="detail-title")
    if title_tag:
        return title_tag.get_text(strip=True)
    
    title_tag = soup.find("h2", class_="media__title")
    if title_tag:
        return title_tag.get_text(strip=True)
        
    title_tag = soup.find("title")
    if title_tag:
        return title_tag.get_text(strip=True).replace(" - detiknews", "").replace(" - detikfinance", "")

    return "Judul Tidak Ditemukan"

def extract_id(url):
    """
    Ekstrak ID berita dari URL, mengatasi format yang berbeda.
    """
    id_match_d = re.search(r"/d-(\d+)", url)
    if id_match_d:
        return id_match_d.group(1)
    id_match_end = re.search(r"-(\d+)$", url)
    if id_match_end:
        return id_match_end.group(1)
    id_match_middle = re.search(r"(\d+)\.html$", url)
    if id_match_middle:
        return id_match_middle.group(1)
        
    return None

# --- FUNGSI UTAMA SCRAPING ---
def berita(categories, pages_per_category=10):
    """Fungsi utama untuk melakukan crawling berita dan menyimpan hasilnya."""
    start_time = time.time()
    session = get_session()
    all_articles_data = []
    processed_links = set()

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/108.0.0.0 Safari/537.36'
    }

    base_urls = {
        "politik": "https://news.detik.com/indeks/berita/",
        "hukum": "https://news.detik.com/indeks/berita/",
        "ekonomi": "https://finance.detik.com/indeks/",
        "detikx": "https://news.detik.com/x/indeks/",
        "hiburan": "https://hot.detik.com/indeks/",
        "internasional": "https://news.detik.com/indeks/berita/",
        "sepakbola": "https://sport.detik.com/sepakbola/indeks/",
        "olahraga": "https://sport.detik.com/indeks/",
        "lingkungan": "https://www.detik.com/tag/lingkungan",
        "otomotif": "https://oto.detik.com/indeks"
    }
    
    categories = list(set(categories))

    for cat in categories:
        current_url = base_urls.get(cat.lower(), f"https://{cat.lower()}.detik.com/indeks/")
        print(f"--- Memulai crawling untuk kategori: {cat} ---")
        
        for page_count in range(1, pages_per_category + 1):
            url = f"{current_url}?page={page_count}"
            if cat.lower() == "lingkungan":
                url = f"https://www.detik.com/tag/lingkungan?page={page_count}"
            
            print_progress(cat, page_count, pages_per_category)

            try:
                r = session.get(url, headers=headers, timeout=15)
                r.raise_for_status()
                soup = BeautifulSoup(r.text, "html.parser")
                article_links = soup.select("a.media__link")
                
                for a in article_links:
                    link = a["href"]
                    if link in processed_links:
                        continue
                    processed_links.add(link)
                    berita_id = extract_id(link)
                    title, content = get_article_content_and_title(session, link)
                    
                    if content:
                        print(f"\n--- Data {len(all_articles_data) + 1} ---")
                        print(f"ID Berita: {berita_id}")
                        print(f"Judul: {title}")
                        print(f"Abstrak (Raw): {content}")
                        print(f"Kategori: {cat}")
                        print("-------------------------------------------\n")

                        all_articles_data.append({
                            "id_berita": berita_id,
                            "judul_berita": title,
                            "isi_berita_original": content,
                            "kategori_berita": cat
                        })
                    time.sleep(random.uniform(1, 3))
            except requests.exceptions.RequestException as e:
                print(f"\n❌ Gagal mengakses {url}: {e}", file=sys.stderr)
                break
    
    df = pd.DataFrame(all_articles_data)
    df.to_csv("crawling_detik_berita.csv", index=False, encoding="utf-8-sig")

    end_time = time.time()
    elapsed = int(end_time - start_time)
    jam, sisa = divmod(elapsed, 3600)
    menit, detik = divmod(sisa, 60)

    print("\n✅ Seluruh data berhasil dikumpulkan!")
    print(f"📊 Total entri: {len(df)}")
    print(f"⏱️ Waktu eksekusi: {jam} jam {menit} menit {detik} detik")
    
    print("\nBerikut adalah 5 entri pertama yang berhasil dikumpulkan:")
    print(df.head())

    return df

if __name__ == '__main__':
    categories = ["politik", "hukum", "ekonomi", "lingkungan", "hiburan", "internasional", "otomotif", "olahraga", "sepakbola"]
    berita(categories, pages_per_category=5)

--- Memulai crawling untuk kategori: ekonomi ---
ekonomi - Page 1/5 [████----------------] 20.00%


--- Data 1 ---
ID Berita: 8656465
Judul: Harga Emas Antam Hari Ini Akhirnya Naik!
Abstrak (Raw): Harga emas Antam akhirnya mengalami kenaikan usai jatuh sangat dalam selama tren pelemahan sejak akhir pekan kemarin. Harga emas Antam 24 karat hari naik Rp 15.000 per gram berada di level Rp 2.625.000 per gram. Berdasarkan situs Logam Mulia Antam, Kamis (10/9/2026), satuan harga emas hari ini yang terkecil ukuran 0,5 gram berada di angka Rp 1.355.000. Sementara harga emas 10 gram dijual dengan harga Rp 25.595.000 dan ukuran emas terbesar yakni 1.000 gram (1 kg) dibanderol Rp 2.550.600.000. Jika ditarik dalam sepekan terakhir, harga emas Antam berada dalam rentang Rp 2.610.000-2.670.000 per gram. Sementara dalam sebulan terakhir, harga emas Antam bergerak dalam rentang Rp 2.610.000-2.768.000 per gram. SCROLL TO CONTINUE WITH CONTENT Harga emas hari ini untuk buyback emas Antam juga naik Rp 15.000 dan berada di level Rp 2.475.000 per gram. Harga buyback adalah jika Anda ingin menjual emas, 


--- Data 2 ---
ID Berita: 8656440
Judul: Dolar AS Menguat Tipis ke Rp 17.522
Abstrak (Raw): Nilai tukar dolar Amerika Serikat (AS) menguat terhadap rupiah pagi ini. Mata uang Paman Sam kini berada di level Rp 17.500-an. Dikutip dari data Bloomberg, Kamis (10/9/2026), nilai tukar dolar AS berada pada level Rp 17.522 atau menguat 0,06%. SCROLL TO CONTINUE WITH CONTENT Selanjutnya, pergerakan dolar AS terhadap sejumlah mata uang lainnya bervariasi namun cenderung stagnan. Nilai tukar dolar AS terhadap dolar Kanada, CHF Swiss France, dan dolar Hong Kong berada pada posisi stagnan. Sementara, dolar AS melemah 0,15% terhadap yen Jepang. Dolar AS juga melemah terhadap won Korea 2,46%.
Kategori: ekonomi
-------------------------------------------




--- Data 3 ---
ID Berita: 8656439
Judul: BULOG Kawal Penyaluran Bantuan Pangan Beras di Cirebon agar Tepat Sasaran
Abstrak (Raw): Perum BULOG terus mengawal penyaluran Bantuan Pangan Beras agar manfaatnya diterima oleh masyarakat yang berhak. Melalui satuan kerja di Cabang Cirebon, BULOG memastikan setiap tahapan distribusi dimonitor secara optimal dalam mendukung program pemerintah untuk menjaga ketahanan pangan sekaligus memenuhi kebutuhan masyarakat. Menanggapi informasi dugaan penyalahgunaan 198 karung beras Bantuan Pangan oleh oknum perangkat Desa Ambit, Kecamatan Waled, Kabupaten Cirebon, BULOG menghormati proses hukum yang sedang berjalan. Pemimpin Wilayah Perum BULOG Jawa Barat, Nurman Susilo menyampaikan penyaluran Bantuan Pangan oleh di Kabupaten Cirebon, khususnya di Desa Ambit Kecamatan Waled, telah dilakukan sesuai ketentuan kepada PBP yang telah ditetapkan pemerintah. SCROLL TO CONTINUE WITH CONTENT "Tim Kami di BULOG Cabang Cirebon telah menyalurkan Bantuan Pangan Beras


--- Data 4 ---
ID Berita: 8656434
Judul: IHSGÂ Menguat Pagi Ini, Tembus 6.700
Abstrak (Raw): Indeks Harga Saham Gabungan (IHSG) dibuka menguat pagi ini. IHSG terus menguat dan menembus level 6.700. Berdasarkan data perdagangan RTI, Kamis (10/9/2026), IHSG pagi ini dibuka pada level 6.688,17. Kemudian nilai indeks saham Garuda ini terus menguat hingga 25,29 poin atau sekitar 0,38% ke level 6.703,43 per pukul 09.05 WIB. SCROLL TO CONTINUE WITH CONTENT Pada level tertinggi IHSG berhasil menyentuh 6.707,91 dan paling rendah hanya di 6.684,76. Mayoritas saham menguat dengan rincian 297 saham naik, 163 saham melemah, dan 228 saham stagnan. Volume perdagangan pagi ini tercatat sudah mencapai 3,78 miliar saham dengan nilai transaksi mencapai Rp 2,48 triliun. Frekuensi perdagangan sebanyak 161.918 kali. IHSG tercatat mengalami kenaikan hingga 0,51% secara mingguan dan naik 6,93% secara bulanan. Namun IHSG sejak awal tahun (year to date) tercatat masih melemah 22,49%.
Kategori: ekonomi
--------

...


--- Data 6 ---
ID Berita: 8656403
Judul: SSIA Pertahankan Target Laba Rp 400 Miliar, JSMR & BMAS Siapkan Aksi Korporasi
Abstrak (Raw): Indeks Harga Saham Gabungan (IHSG) kembali berakhir di zona merah pada perdagangan Rabu (9/9/2026). IHSG turun 0,12% ke level 6.678,20. Sejumlah saham menjadi penggerak perdagangan. AMMN naik 5,14%, disusul IMPC yang menguat 9,51% dan AADI sebesar 5,96%. Di sisi lain, BBCA turun 2,25%, BMRI melemah 0,90%, sementara ASII terkoreksi 1,62%. Tekanan juga datang dari investor asing. Di pasar reguler, tercatat penjualan bersih asing sebesar Rp621,40 miliar. Jika memperhitungkan seluruh pasar, penjualan bersih asing mencapai Rp438,52 miliar. SCROLL TO CONTINUE WITH CONTENT Pergerakan sektoral turut menunjukkan tekanan. Sebanyak 6 dari 11 sektor berakhir melemah. Sektor Healthcare menjadi yang mengalami penurunan paling dalam, yakni 1,33%. Sebaliknya, Transportation & Logistic mencatat penguatan terbesar sebesar 2,41%. Dari bursa Amerika Serikat, tiga indeks u


--- Data 7 ---
ID Berita: 8656411
Judul: Ekonomi Batam Tumbuh 7,28%, Ini Penopangnya
Abstrak (Raw): Pertumbuhan ekonomi Batam pada kuartal II 2026 tercatat 7,28% secara yoy. Angka ini lebih tinggi dibanding pertumbuhan ekonomi Provinsi Kepulauan Riau (Kepri) sebesar 6,99% dan pertumbuhan ekonomi nasional sebesar 5,29%. Aktivitas ekonomi Batam sendiri ditopang sejumlah sektor, antara lain industri, investasi, perdagangan, dan jasa. Kepala BP Batam, Amsakar Achmad, mengatakan capaian pertumbuhan tersebut menunjukkan kepercayaan dunia usaha terhadap Batam yang terus menguat. SCROLL TO CONTINUE WITH CONTENT "Pertumbuhan 7,28% adalah hasil kerja bersama. Tugas kita sekarang adalah menjaga momentum melalui pelayanan yang semakin cepat, infrastruktur yang semakin baik, dan iklim investasi yang semakin kompetitif," kata Amsakar dalam keterangan tertulis, Kamis (10/9/2026). Menurutnya, keberlanjutan pertumbuhan ekonomi Batam perlu didukung dengan peningkatan kualitas pelayanan, pembangunan inf


--- Data 8 ---
ID Berita: 8656368
Judul: Harga Minyak Tembus US$ 101 per Barel!
Abstrak (Raw): Harga minyak mentah dunia terus melonjak dan kini kembali menembus level US$ 101 per barel. Ini merupakan harga penutupan tertinggi sejak Mei lalu. Melansir CNBC, Kamis (10/9/2026), lonjakan harga minyak ini didorong oleh meningkatnya pertempuran antara Amerika Serikat (AS) melawan Iran di Teluk Persia, membuat pelaku pasar semakin khawatir akan penurunan ekspor energi melalui Selat Hormuz. Pada akhir perdagangan Rabu (10/9) kontrak berjangka Brent yang kerap menjadi patokan global naik 3,4% menjadi US$ 101,21 per barel. Sementara harga minyak mentah AS West Texas Intermediate (WTI) juga naik hingga 3,3% dan ditutup pada US$ 96,05 per barel. SCROLL TO CONTINUE WITH CONTENT Untuk diketahui, pasukan militer AS tercatat berhasil menghancurkan lima kapal tanker minyak mentah Iran pada Selasa (8/9). Hal ini dilakukan sebagai balasan atas upaya serangan Iran terhadap kapal perang milik Amerika, me


--- Data 9 ---
ID Berita: 8656331
Judul: Penumpang Makin Padat, KCI Tambah 13 Perjalanan KRL Bogor
Abstrak (Raw): PT Kereta Commuter Indonesia (KCI) atau KAI Commuter melakukan evaluasi atas pola operasi perjalanan sejak melakukan penyesuaian perjalanan Commuter Line Bogor pada Senin 7 September 2026 karena perawatan sarana KRL VP Corporate Secretary KAI Commuter, Karina Amanda mengatakan berdasarkan hasil evaluasi tersebut, mulai hari ini, Kamis (10/9), pihaknya akan menambah kembali 13 perjalanan Commuter Line Bogor alias Red Line. Penambahan ini merupakan hasil evaluasi atas kepadatan pengguna yang terjadi imbas penyesuaian 17 perjalanan Commuter Line Bogor. "Langkah ini diambil dengan melakukan pengaturan pola peredaran sarana Commuter Line di seluruh lintas layanan untuk mengoptimalkan jumlah jadwal perjalanan," kata Karina dalam keterangan resminya, Kamis (10/9/2026). SCROLL TO CONTINUE WITH CONTENT Bersama dengan itu, Karina menambahkan KAI Commuter akan melakukan rekayasa pola

KeyboardInterrupt: 